In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
ALPHA=config.ALPHA_PRIMARY
IOT_RAW = config.DATASETS_DIR/'ciciot2023'
for d in (config.PROC_DIR, config.REPORTS_DIR, config.INTERIM_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('ready:', os.getcwd())
print('iot raw dir exists:', IOT_RAW.exists(), '|', IOT_RAW)


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research
iot raw dir exists: False | /content/drive/MyDrive/NIDS_Datasets/ciciot2023


In [3]:
# =============================================================================
# Cell 2 - DISCOVER the CIC-IoT-2023 release and inventory its labels.
# The preregistration (section 3) names this dataset "modern heterogeneous,
# family holdout" and section 10.1 fixes family-holdout combinations with S_cov
# and S_sup as the analysis variables. Nothing here computes coverage: this cell
# only establishes what is on disk and what the label inventory actually is, so
# the ladder and focal class can be fixed against reality rather than assumption.
# =============================================================================
files = sorted(glob.glob(str(IOT_RAW/'**'/'*.csv'), recursive=True))
print(f'CSV files found: {len(files)}')
if not files:
    print('\nNo CSVs under', IOT_RAW)
    print('Contents of the datasets dir:')
    for p in sorted(config.DATASETS_DIR.glob('*')): print('  ', p.name)
    raise SystemExit('place the CIC-IoT-2023 CSV release under ' + str(IOT_RAW))
print('first 3:', [Path(f).name for f in files[:3]])

head = pd.read_csv(files[0], nrows=5)
print(f'\ncolumns ({len(head.columns)}):', list(head.columns))
LABEL_COL = next((c for c in head.columns if c.strip().lower() in ('label','labels','attack','class')), None)
assert LABEL_COL, 'no label column found; inspect the columns above'
print('label column:', LABEL_COL)

# inventory labels across ALL files but reading only the label column (cheap)
from collections import Counter
cnt = Counter()
for i,f in enumerate(files):
    s = pd.read_csv(f, usecols=[LABEL_COL])[LABEL_COL].astype(str).str.strip()
    cnt.update(s.value_counts().to_dict())
    if (i+1) % 25 == 0: print(f'  scanned {i+1}/{len(files)} files')
inv = pd.Series(cnt).sort_values(ascending=False)
print(f'\ntotal rows: {inv.sum():,} | distinct labels: {len(inv)}')
print(inv.to_string())


CSV files found: 0

No CSVs under /content/drive/MyDrive/NIDS_Datasets/ciciot2023
Contents of the datasets dir:
   cicids2017-wtmc2021
   nsl-kdd
   ugr16


SystemExit: place the CIC-IoT-2023 CSV release under /content/drive/MyDrive/NIDS_Datasets/ciciot2023

In [ ]:
# =============================================================================
# Cell 3 - TAXONOMY: map each specific attack label to (class, subtype).
# Same convention as CIC-IDS2017 (Amendment 5 A5.2): the broad attack family is
# the CLASS on which Mondrian calibration and the focal-class rule operate; the
# specific attack is the SUBTYPE that drives S_sup under family/variant holdout.
# Mapping is derived from the label string itself, so it adapts to whatever the
# release actually contains rather than hardcoding a list.
# =============================================================================
def to_family(lbl):
    L = lbl.strip()
    low = L.lower()
    if low.startswith('benign'):            return 'Benign'
    if low.startswith('ddos'):              return 'DDoS'
    if low.startswith('dos'):               return 'DoS'
    if low.startswith('mirai'):             return 'Mirai'
    if low.startswith('recon') or 'scan' in low or 'sweep' in low or 'discovery' in low:
        return 'Recon'
    if 'spoofing' in low:                   return 'Spoofing'
    if 'bruteforce' in low or 'dictionary' in low: return 'BruteForce'
    # Web is the residual family. Every label reaching it is listed explicitly below so
    # that a label added by a future release cannot be silently absorbed into Web.
    return 'Web'

def to_subtype(lbl):
    L = lbl.strip()
    for sep in ('-', '_'):
        if sep in L:
            head, rest = L.split(sep, 1)
            if head.lower() in ('ddos','dos','mirai','recon'): return rest
    return L

tax = pd.DataFrame({'label': inv.index, 'n': inv.values})
tax['family']  = tax['label'].map(to_family)
tax['subtype'] = tax['label'].map(to_subtype)
fam = (tax.groupby('family')
          .agg(rows=('n','sum'), n_subtypes=('subtype','nunique'),
               subtypes=('subtype', lambda s: ', '.join(sorted(set(s))[:6])))
          .sort_values('rows', ascending=False))
print('FAMILY INVENTORY (family = class, specific attack = subtype):')
print(fam.to_string())
print('\nshare of all rows by family:')
print((fam['rows']/fam['rows'].sum()).round(5).to_string())
print('\nA family needs >= 2 subtypes to support a variant-holdout ladder in which')
print('the focal class is present in BOTH source and target (the CIC-IDS2017 lesson:')
print('holding a whole family out of source makes it infeasible under SHC and')
print('therefore unusable as the focal class).')
print('\nfamilies with >= 2 subtypes:', sorted(fam[fam.n_subtypes>=2].index.tolist()))

# guard: anything landing in the residual Web family must be a known web/host attack.
KNOWN_WEB = {'BrowserHijacking','CommandInjection','SqlInjection','XSS',
             'Uploading_Attack','Backdoor_Malware'}
residual = set(tax[tax.family=='Web']['label']) - KNOWN_WEB
if residual:
    print('\nWARNING: labels absorbed into the residual Web family that are not on the known list:')
    for r in sorted(residual): print('   ', r)
    print('   Inspect these before proceeding; the taxonomy may need an explicit rule.')
else:
    print('\nresidual-family guard: no unexpected labels absorbed into Web')


In [ ]:
# =============================================================================
# Cell 4 - BUILD the working frame: stratified subsample, features, partitions.
# CIC-IoT-2023 is far larger than the compute budget, so rows are subsampled per
# label with a cap (section 5 caps training at 2M rows). Sampling is per-label so
# rare families survive; the realised counts are recorded.
# =============================================================================
PER_LABEL_CAP = 60000          # target rows per specific attack label

# Memory-safe two-pass subsample. Cell 2 already counted every label across the
# whole release, so the keep-fraction per label is known in advance and each file
# is thinned as it loads. The full 40M+ row frame is never held in memory, and the
# sample stays unbiased within each label.
keep_frac = {lbl: min(1.0, PER_LABEL_CAP / float(n)) for lbl, n in inv.items()}
print('labels kept in full:', sum(1 for v in keep_frac.values() if v >= 1.0),
      '| thinned:', sum(1 for v in keep_frac.values() if v < 1.0))

parts=[]
for i, f in enumerate(files):
    d = pd.read_csv(f)
    d[LABEL_COL] = d[LABEL_COL].astype(str).str.strip()
    rng_f = np.random.default_rng(int(hashlib.sha256(Path(f).name.encode()).hexdigest(), 16) % (2**32))
    fr = d[LABEL_COL].map(keep_frac).fillna(1.0).to_numpy(dtype=float)
    parts.append(d[rng_f.random(len(d)) < fr])
    del d
    if (i+1) % 25 == 0: print(f'  loaded+thinned {i+1}/{len(files)}')
iot = pd.concat(parts, ignore_index=True).reset_index(drop=True); del parts
iot['family']  = iot[LABEL_COL].map(to_family)
iot['subtype'] = iot[LABEL_COL].map(to_subtype)
print('subsampled:', iot.shape)
print(f'\nrealised rows per label (target cap {PER_LABEL_CAP}):')
print(iot[LABEL_COL].value_counts().to_string())
print('\nrealised family counts:')
print(iot['family'].value_counts().to_string())

DROP_PAT = ('unnamed', 'index', 'id', 'timestamp', 'time', 'flow_id', 'src_ip', 'dst_ip')
FEATS = [c for c in iot.columns
         if c not in (LABEL_COL,'family','subtype')
         and pd.api.types.is_numeric_dtype(iot[c])
         and not any(k == c.strip().lower() or c.strip().lower().startswith(k+' ') or
                     c.strip().lower().startswith('unnamed') for k in DROP_PAT)]
dropped = [c for c in iot.columns
           if c not in FEATS and c not in (LABEL_COL,'family','subtype')]
print(f'\nnumeric features kept: {len(FEATS)}')
print('columns excluded from features:', dropped if dropped else 'none')

# a row-order or index column would leak label structure; check the survivors
const = [c for c in FEATS if iot[c].nunique(dropna=False) <= 1]
if const:
    print('constant features dropped:', const)
    FEATS = [c for c in FEATS if c not in const]
print(f'final feature count: {len(FEATS)}')
bad = [c for c in FEATS if not np.isfinite(iot[c].to_numpy(dtype=float, na_value=np.nan)).all()]
print('features containing non-finite values:', len(bad))
iot[FEATS] = iot[FEATS].replace([np.inf,-np.inf], np.nan)
iot[FEATS] = iot[FEATS].fillna(iot[FEATS].median(numeric_only=True))

def strat(df, fr, seed, col):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float)
    big=nm[int(np.argmax(ff))]; a=pd.Series(index=df.index, dtype=object)
    for _, s in df.groupby(col, sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)] += n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a

IOT_PARTITION_SEED = 20260726
iot['partition'] = strat(iot, config.SPLIT_FRACTIONS, IOT_PARTITION_SEED, LABEL_COL).values
print('\npartition sizes:'); print(iot['partition'].value_counts().to_string())
print('\npartitions disjoint and exhaustive:', iot['partition'].notna().all())


In [ ]:
# =============================================================================
# Cell 5 - FEASIBILITY TABLE and FOCAL-CLASS RECORD. This is the gate.
# Section 7.6: a class needs ceil(1/alpha)-1 calibration points. Section 12: the
# focal class is the rarest attack class satisfying that rule in the SOURCE
# calibration pool, recorded BEFORE any coverage outcome is examined.
# Additional binding constraint carried from the CIC-IDS2017 experience: the
# focal class must also have >= 2 subtypes, otherwise no variant-holdout ladder
# can shift it while keeping it feasible under SHC.
# =============================================================================
pool = iot[iot.partition=='source_cal_pool']
need = conformal.min_calib_n(ALPHA)
rows=[]
for famname, g in iot.groupby('family'):
    n_pool = int((pool['family']==famname).sum())
    rows.append({'family':famname,'rows_total':int(len(g)),'n_source_cal_pool':n_pool,
                 'min_needed':need,'feasible':bool(n_pool>=need),
                 'n_subtypes':int(g['subtype'].nunique()),
                 'ladder_capable':bool(n_pool>=need and g['subtype'].nunique()>=2)})
feas=pd.DataFrame(rows).sort_values('n_source_cal_pool')
print(f'FEASIBILITY at alpha={ALPHA} (need {need} source calibration points per class):')
print(feas.to_string(index=False))

cand = feas[(feas.family!='Benign') & (feas.ladder_capable)].sort_values('n_source_cal_pool')
assert len(cand)>0, 'no attack family is both feasible and multi-subtype; ladder must be redesigned'
FOCAL = cand.iloc[0]['family']
print(f'\nFOCAL CLASS (rarest feasible multi-subtype attack family): {FOCAL}')
print(f'  source calibration pool rows: {int(cand.iloc[0]["n_source_cal_pool"])}')
print(f'  subtypes: {sorted(iot[iot.family==FOCAL]["subtype"].unique())}')
excluded = feas[(feas.family!="Benign") & (~feas.feasible)]['family'].tolist()
print(f'\nEXCLUDED as infeasible (reported, never forced): {excluded or "none"}')

record = {'dataset':'ciciot2023','alpha':ALPHA,'min_calib_needed':need,
          'focal_class':FOCAL,
          'focal_selection_rule':'rarest attack family satisfying section 7.6 in the source '
                                 'calibration pool AND holding >=2 subtypes so a variant-holdout '
                                 'ladder can shift it while keeping it feasible under SHC',
          'focal_subtypes':sorted(iot[iot.family==FOCAL]['subtype'].unique().tolist()),
          'excluded_infeasible':excluded,
          'per_label_cap':PER_LABEL_CAP,'partition_seed':IOT_PARTITION_SEED,
          'label_column':LABEL_COL,'n_features':len(FEATS),
          'feasibility_table':feas.to_dict('records'),
          'recorded_before_any_coverage':True}
(config.REPORTS_DIR/'focal_class_record_ciciot2023.json').write_text(json.dumps(record,indent=2,default=str))
feas.to_csv(config.REPORTS_DIR/'feasibility_binding_ciciot2023.csv',index=False)
tax.to_csv(config.REPORTS_DIR/'label_taxonomy_ciciot2023.csv',index=False)
print('\nrecorded to reports/focal_class_record_ciciot2023.json')


In [ ]:
# =============================================================================
# Cell 6 - persist the prepared frame (gitignored) + fingerprint, then commit
# the gate artefacts. Coverage is computed in the NEXT notebook, so this commit
# timestamps the focal class ahead of any result.
# =============================================================================
IOT_PROC = config.PROC_DIR/'ciciot2023_prepared.parquet'
keep = FEATS + [LABEL_COL,'family','subtype','partition']
iot[keep].to_parquet(IOT_PROC, index=False)
h = hashlib.sha256(IOT_PROC.read_bytes()).hexdigest()
print('prepared frame:', IOT_PROC, '|', iot.shape, '| sha256', h[:16])
(config.REPORTS_DIR/'ciciot2023_prepared_fingerprint.json').write_text(json.dumps(
    {'path':str(IOT_PROC),'rows':int(len(iot)),'cols':int(len(keep)),
     'sha256':h,'features':FEATS,'label_column':LABEL_COL}, indent=2))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb31: CIC-IoT-2023 inventory, taxonomy, partitions, feasibility and focal-class record (before any coverage)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
